In [ ]:
pip install pandas numpy scikit-learn matplotlib

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [4]:
# PHASE A: LOAD DATASET & DEFINE TARGET
print("Loading 'student_dropout_dataset_v3.csv'...")
df = pd.read_csv('student_dropout_dataset_v3.csv')
df.head()

Loading 'student_dropout_dataset_v3.csv'...


,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,4,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,5,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


In [10]:
# Define X (features) and y (target). Drop 'Student_ID' as it's not a feature.
X = df.drop(['Student_ID'], axis=1)
y = df['Dropout']
X.head()

,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


In [7]:
# PHASE B: DATA CLEANING & ENCODING
print("\nStarting Data Cleaning and Encoding...")

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')
categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_imputer, numeric_cols),
        ('cat', categorical_encoder, categorical_cols)
    ]
)

X_processed = preprocessor.fit_transform(X)
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
feature_names = np.concatenate([numeric_cols, cat_names])
X_clean = pd.DataFrame(X_processed, columns=feature_names)
X_clean.head()


Starting Data Cleaning and Encoding...


,Age,Family_Income,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Stress_Index,GPA,Semester_GPA,CGPA,...,Department_Arts,Department_Business,Department_CS,Department_Engineering,Department_Science,Parental_Education_Bachelor,Parental_Education_High School,Parental_Education_Master,Parental_Education_PhD,Parental_Education_nan
0,22.1,25000.0,3.36,86.1,2.0,20.4,5.5,0.96,0.90,0.90,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,20.7,25000.0,4.30,68.0,2.0,44.0,6.8,1.28,1.20,1.19,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,22.4,40183.0,4.40,70.9,0.0,48.9,5.5,1.68,1.32,1.32,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,24.4,29740.5,4.00,82.2,2.0,38.6,5.5,1.78,1.77,1.77,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,20.5,25319.0,4.19,75.7,1.0,23.0,7.0,1.48,0.91,0.87,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [ ]:
# PHASE C: STRATIFIED SPLITTING (UPDATED)
print("\nPerforming Stratified Splitting...")

# Split 1: 40% Main Model (Set-1), 60% remainder
X_set1, X_remainder, y_set1, y_remainder = train_test_split(
    X_clean, y, test_size=0.60, stratify=y, random_state=42
)

# Split 2: Divide remainder into Campus 1 and Campus 2
X_set2, X_set3, y_set2, y_set3 = train_test_split(
    X_remainder, y_remainder, test_size=0.50, stratify=y_remainder, random_state=42
)

# Split 3: Create Train/Test splits (80/20) for each Set
X_set1_train, X_set1_test, y_set1_train, y_set1_test = train_test_split(X_set1, y_set1, test_size=0.20, stratify=y_set1, random_state=42)
X_set2_train, X_set2_test, y_set2_train, y_set2_test = train_test_split(X_set2, y_set2, test_size=0.20, stratify=y_set2, random_state=42)
X_set3_train, X_set3_test, y_set3_train, y_set3_test = train_test_split(X_set3, y_set3, test_size=0.20, stratify=y_set3, random_state=42)


Performing Stratified Splitting...


In [22]:
# PHASE D: DOCUMENTATION & EXPORT
import pandas as pd
from IPython.display import display

print("\n" + "="*25)
print("DATA SPLIT SUMMARY")
print("="*25)
print(f"Total Original Dataset: {len(X_clean)} rows\n")

split_summary = {
    'Environment': ['Set-1 (Main Model)', 'Set-2 (Campus 1)', 'Set-3 (Campus 2)'],
    'Total Rows': [len(X_set1), len(X_set2), len(X_set3)],
    'Training Rows (80%)': [len(X_set1_train), len(X_set2_train), len(X_set3_train)],
    'Testing Rows (20%)': [len(X_set1_test), len(X_set2_test), len(X_set3_test)]
}
df_splits = pd.DataFrame(split_summary)

print("--- Data Split Summary ---")
display(df_splits)

print("\n")

strat_summary = {
    'Environment': ['Set-1 (Main Model)', 'Set-2 (Campus 1)', 'Set-3 (Campus 2)'],
    'No Dropout (0)': [
        f"{y_set1.value_counts(normalize=True).get(0, 0) * 100:.2f}%",
        f"{y_set2.value_counts(normalize=True).get(0, 0) * 100:.2f}%",
        f"{y_set3.value_counts(normalize=True).get(0, 0) * 100:.2f}%"
    ],
    'Dropout (1)': [
        f"{y_set1.value_counts(normalize=True).get(1, 0) * 100:.2f}%",
        f"{y_set2.value_counts(normalize=True).get(1, 0) * 100:.2f}%",
        f"{y_set3.value_counts(normalize=True).get(1, 0) * 100:.2f}%"
    ]
}
df_strat = pd.DataFrame(strat_summary)

print("--- Stratification Check (Dropout Distribution) ---")
display(df_strat)

print("="*45)


DATA SPLIT SUMMARY
Total Original Dataset: 10000 rows

--- Data Split Summary ---


,Environment,Total Rows,Training Rows (80%),Testing Rows (20%)
0,Set-1 (Main Model),4000,3200,800
1,Set-2 (Campus 1),3000,2400,600
2,Set-3 (Campus 2),3000,2400,600




--- Stratification Check (Dropout Distribution) ---


,Environment,No Dropout (0),Dropout (1)
0,Set-1 (Main Model),76.45%,23.55%
1,Set-2 (Campus 1),76.47%,23.53%
2,Set-3 (Campus 2),76.47%,23.53%


In [ ]:
X_set1.assign(Dropout=y_set1).to_csv('main_cloud_data.csv', index=False)
X_set2.assign(Dropout=y_set2).to_csv('campus1_data.csv', index=False)
X_set3.assign(Dropout=y_set3).to_csv('campus2_data.csv', index=False)

print("\nSUCCESS! 3 CSV files generated.")